# Forge F01 — R2 | Python 3.13 uyumlu başlangıç
T4 GPU ve ağ erişimi kontrol edilir. Colab sürüm ayarını değiştirmeyin. Ayrı Python 3.10 kurulacaktır. Resmi TripoSG demo deneyi; GPU ve ağırlık indirme başarısız olursa durur. Ticari kullanım için RMBG lisansını ayrıca doğrulayın.

In [ ]:
import sys, json, subprocess, pathlib, time, urllib.request
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü değiştir → T4 GPU → Kaydet."
gpu = torch.cuda.get_device_properties(0)
print("CUDA:", torch.cuda.is_available(), "| GPU:", gpu.name, "| VRAM GiB:", round(gpu.total_memory/2**30, 2))
assert gpu.total_memory >= 8 * 2**30, "TripoSG için en az 8 GiB VRAM gerekli."
x = torch.arange(1024, device="cuda", dtype=torch.float32)
assert float(x.sum().item()) == 523776.0
torch.cuda.synchronize()
print("GERÇEK CUDA HESABI GEÇTİ")
for url in ["https://github.com/VAST-AI-Research/TripoSG", "https://huggingface.co/api/models/VAST-AI/TripoSG", "https://huggingface.co/api/models/briaai/RMBG-1.4"]:
    with urllib.request.urlopen(url, timeout=25) as response:
        print(response.status, url)
print("Python:",sys.version,"PyTorch:",torch.__version__,"CUDA build:",torch.version.cuda)


## 2. Ayrı Python 3.10 ve gerçek CUDA eklentisi
Resmi TripoSG Space'in yayımladığı DISO 0.1.4 wheel SHA256 doğrulaması. CUDA/PyTorch ABI uyumsuzluğu olursa hata saklanmaz.

In [ ]:
import os, sys, subprocess, pathlib, hashlib, urllib.request, shutil
root=pathlib.Path("/content/forge-f01-triposg-r2")
venv=pathlib.Path("/content/forge-f01-py310")
rev="fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c"
env=dict(os.environ,HF_HUB_DISABLE_XET="1",PYTHONNOUSERSITE="1",PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
env.pop("PYTHONPATH",None)
def run(args,cwd=None,timeout=1800):
    print("RUN:", " ".join(map(str,args)), flush=True)
    subprocess.run(list(map(str,args)),cwd=cwd,env=env,check=True,timeout=timeout)
assert torch.cuda.is_available()
assert shutil.disk_usage("/content").free>12*2**30, "12 GiB boş disk gerekli"
# Keep Colab kernel Python untouched; create an independent Python 3.10 process.
run([sys.executable,"-m","pip","install","uv==0.9.26"])
run([sys.executable,"-m","uv","python","install","3.10"])
if not (venv/"bin/python").exists():
    run([sys.executable,"-m","uv","venv","--python","3.10","--seed",venv])
py=venv/"bin/python"
run([py,"-c","import sys; assert sys.version_info[:2]==(3,10); print(sys.version)"])
run([py,"-m","pip","install","torch==2.5.1","torchvision==0.20.1","--index-url","https://download.pytorch.org/whl/cu124"])
run([py,"-m","pip","install","numpy==1.26.4","scipy==1.11.4","diffusers==0.32.2","transformers==4.49.0","huggingface-hub==0.28.1","safetensors==0.5.3","accelerate==1.4.0","peft==0.14.0","einops==0.8.1","opencv-python-headless==4.10.0.84","trimesh==4.5.3","omegaconf==2.3.0","scikit-image==0.24.0","jaxtyping==0.2.38","typeguard==2.13.3","pymeshlab==2022.2.post4","Pillow==11.1.0"])
if not root.exists():
    run(["git","clone","https://github.com/VAST-AI-Research/TripoSG.git",root])
run(["git","-C",root,"checkout","--detach",rev])
assert subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()==rev
# Publisher-provided CPython 3.10 binary; verify its published SHA256.
wheel=pathlib.Path("/content/diso-0.1.4-cp310-cp310-linux_x86_64.whl")
urllib.request.urlretrieve("https://huggingface.co/spaces/VAST-AI/TripoSG/resolve/d51c5f2f4d5182f925a403b286d1e9faf519aaa2/"+wheel.name,wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest()=="8814512bb12ef32fe2aa8572db76cf73ee5e7f604764a37337bfd087e667dce8", "DISO wheel checksum mismatch"
run([py,"-m","pip","install","--no-deps",wheel])
run([py,"-m","pip","check"])
# Verify real CUDA and compiled extension, not only import.
run([py,"-c","import torch; from diso import DiffDMC; assert torch.cuda.is_available(); x=torch.arange(1024,device='cuda'); assert x.sum().item()==523776; d=DiffDMC(dtype=torch.float32).cuda(); print('PY310 CUDA OK',torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))"])
run([py,"-c","from scripts.inference_triposg import run_triposg; print('TripoSG import OK')"],cwd=root)
print("AYRI PYTHON 3.10 KURULUMU VE CUDA KONTROLÜ GEÇTİ")


## 3. Gerçek TripoSG çıkarımı
Ağırlıklar safetensors ile indirilir. Tek demo çıktısı F01 kalite kabulü değildir.

In [ ]:
# Run model in the separate environment; keep the original Colab kernel untouched.
runner="import json, hashlib, time, pathlib, shutil, sys\nimport torch, numpy as np, trimesh\nfrom huggingface_hub import snapshot_download\nfrom safetensors.torch import load_file\nfrom scripts.inference_triposg import run_triposg, BriaRMBG, TripoSGPipeline\nroot=pathlib.Path.cwd()\nout=pathlib.Path(\"/content/forge-f01-gpu-evidence-r2\");out.mkdir(exist_ok=True)\nsrc=root/\"assets/example_data/hjswed.png\"\nassert src.is_file() and torch.cuda.is_available()\nt=time.monotonic()\ntripo_rev=\"2c1c516d22d58db486a058d98d31bb6177344e06\"\nbria_rev=\"9921b8c152a4b24e3f559592d800f42ed7e06aeb\"\ntripo=snapshot_download(\"VAST-AI/TripoSG\",revision=tripo_rev,allow_patterns=[\"*.json\",\"*.safetensors\",\"*.safetensors.index.json\",\"*.txt\"],local_dir=str(root/\"pretrained_weights/TripoSG-safe\"))\nbria=snapshot_download(\"briaai/RMBG-1.4\",revision=bria_rev,allow_patterns=[\"model.safetensors\",\"config.json\",\"README.md\"],local_dir=str(root/\"pretrained_weights/RMBG-safe\"))\n# Only safetensors weights; no pickle fallback or remote Python code.\nrmbg=BriaRMBG()\nrmbg.load_state_dict(load_file(str(pathlib.Path(bria)/\"model.safetensors\")),strict=True)\nrmbg=rmbg.eval().to(\"cuda\")\npipe=TripoSGPipeline.from_pretrained(tripo,use_safetensors=True,local_files_only=True,torch_dtype=torch.float16).to(\"cuda\")\nm=run_triposg(pipe,str(src),rmbg,seed=42,num_inference_steps=50,faces=-1)\nassert len(m.faces)>100 and np.isfinite(m.vertices).all() and np.all(m.extents>0)\ndest=out/\"actual_triposg_demo.glb\";m.export(dest,include_normals=True)\nq=trimesh.load(dest,force=\"mesh\",process=False);assert len(q.faces)==len(m.faces)\nshutil.copy2(src,out/\"source.png\")\nsha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()\nreport=dict(actual_inference=True,engine=\"TripoSG\",model_revision=tripo_rev,background_revision=bria_rev,gpu=torch.cuda.get_device_name(0),python=sys.version,torch=torch.__version__,source_sha256=sha(src),glb_sha256=sha(dest),faces=len(m.faces),vertices=len(m.vertices),watertight=bool(m.is_watertight),seconds=time.monotonic()-t,visual_accepted=False,editor_accepted=False,benchmark_10_completed=False)\n(out/\"report.json\").write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n"
script=root/"forge_actual_inference.py";script.write_text(runner)
run([py,script],cwd=root,timeout=1800)
out=pathlib.Path("/content/forge-f01-gpu-evidence-r2")
assert (out/"actual_triposg_demo.glb").is_file() and (out/"report.json").is_file()
archive=shutil.make_archive("/content/Forge_F01_GPU_Evidence_R2","zip",out)
from google.colab import files
files.download(archive)
